# Séance 13 — Gouvernance des modèles, équité et risque de crédit (IFRS 9)
*Session 13 — Model governance, fairness and credit risk (IFRS 9)*

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noe2019/fraude-360-cours/blob/main/cours/laboratoires/13_gouvernance_credit.ipynb)

**Cours :** Science des données appliquée : détection de la fraude transactionnelle (45 h) · **Chargé de cours :** Noé Carème Fouotsa Manfouo · **Durée :** 3 h

| Parcours de formation | Cette séance |
|---|---|
| Gestion — UdeS École de gestion (SIMQG), Bishop's Williams School | ● séance complète |
| Informatique — UdeS maîtrise/bac en informatique (IA et science des données) | ○ démonstration |
| Cégep / formation continue — Cégep de Sherbrooke (AEC), UdeS formation continue | — hors parcours |

**À la fin de cette séance, vous saurez :**

1. lire une fiche de modèle (model card) et la relier à la ligne directrice E-23 du BSIF ;
2. appliquer la règle des quatre cinquièmes à des taux de blocage et de faux positifs ;
3. calculer le poids de la preuve (WOE) et la valeur d'information (IV) d'une grille de score ;
4. calculer une perte de crédit attendue (ECL) d'étape 1 et d'étape 2.

**Mode d'emploi :** exécutez les cellules dans l'ordre (`Maj + Entrée`). Complétez les cellules `# VOTRE CODE ICI`, puis
lancez la cellule ✅ qui suit pour obtenir une rétroaction immédiate. Chaque exercice indique le ou les parcours concernés :
🟢 tous · 🟣 Gestion + Informatique · 🔷 Informatique · 🔶 Gestion. Les exercices 🔷/🔶 sont facultatifs pour les autres parcours.

In [ ]:
# --- Configuration (une fois par session) ----------------------------------------
import sys, pathlib, subprocess, shutil
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    if not pathlib.Path("fraude-360-cours").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/noe2019/fraude-360-cours.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "fraude-360-cours/cours/requirements-colab.txt"], check=True)
    if shutil.which("java") is None:   # PySpark a besoin d'un Java 11, 17 ou 21
        subprocess.run("apt-get -qq install -y openjdk-17-jre-headless > /dev/null", shell=True, check=True)
    RACINE = pathlib.Path("fraude-360-cours").resolve()
else:
    ici = pathlib.Path.cwd().resolve()
    RACINE = ici.parents[1] if ici.name == "laboratoires" else ici
sys.path.insert(0, str(RACINE / "cours"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from outils_cours import preparer, config, table, verifier, verifier_proche, image, lire_json
pd.set_option("display.max_columns", 40)
print("Racine du projet :", RACINE)

In [ ]:
cfg = preparer("carte-modele")
from IPython.display import Markdown, display
carte = (cfg.chemin("artefacts") / "gouvernance" / "carte_modele.md").read_text(encoding="utf-8")
display(Markdown(carte[:3500] + "\n\n*(…)*"))

In [ ]:
from fraude360.gouvernance import equite
eq = lire_json(cfg.chemin("artefacts") / "gouvernance" / "rapport_equite.json")
print(eq["regle"], "|", eq["conclusion"])
pd.DataFrame(eq["axes"][0]["sous_groupes"]).T

### Exercice 13.1 — 🟢 Tous les parcours

Avec `dec = table("gold", "decisions")`, calculez par `canal` le taux de blocage (`bloquee`), puis `ratio_impact` =
taux du canal / taux du canal **le plus nombreux**. Ajoutez `a_justifier` = ratio hors de [0,80; 1,25]. Résultat dans `impact`.

In [ ]:
# VOTRE CODE ICI
dec = table("gold", "decisions")
impact = ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
_axe = next(a for a in eq["axes"] if a["axe"] == "canal")
_ref = pd.Series({k: v["ratio_impact"] for k, v in _axe["sous_groupes"].items()})
verifier(np.allclose(impact.loc[_ref.index, "ratio_impact"].round(3), _ref, atol=1e-3), "Mêmes ratios que le rapport d'équité")

**❓ Question (🔶 Gestion — décision d'affaires) —** Un canal a un ratio d'impact de 3, mais ses taux de **faux positifs** sont comparables aux autres. Est-ce une discrimination ? Que présentez-vous à la deuxième ligne de défense ?

*Votre réponse ici.*

## Volet crédit : grille de score et IFRS 9

In [ ]:
from fraude360.credit import pipeline_credit, ifrs9
rap_credit = pipeline_credit.executer(cfg)          # portefeuille simulé, grille WOE, validation, ECL (≈ 1 min)
print(rap_credit["validation"]["conclusion"])
image(cfg.chemin("artefacts") / "credit" / "40_credit_validation.png")
woe = pd.read_csv(cfg.chemin("artefacts") / "credit" / "tables_woe.csv")
woe.head(8)

### Exercice 13.2 — 🟢 Tous les parcours

Pour la variable `score_bureau` de `woe`, recalculez à partir de `nb_bons` et `nb_mauvais` :
`part_bons = (nb_bons + 0.5) / (total_bons + 0.5 × k)` et de même pour `part_mauvais` (k = nombre de classes, correction de Laplace),
`woe = ln(part_bons / part_mauvais)` et la valeur d'information `iv = Σ (part_bons − part_mauvais) × woe`.

In [ ]:
# VOTRE CODE ICI
t = woe[woe.variable == "score_bureau"].copy()
iv = ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
verifier(np.allclose(t["woe_calc"], t["woe"]), "Vos WOE doivent égaler ceux de la grille")
verifier_proche(iv, t["iv_partiel"].sum(), 1e-9)

### Exercice 13.3 — 🔷 Informatique — approfondissement

Un prêt : PD 12 mois = 2 %, LGD = 45 %, exposition = 10 000 $, durée résiduelle = 60 mois, taux d'actualisation
`ifrs9.TAUX_ACTUALISATION`. PD marginale de l'année t (t = 1…5) = (1 − PD)^(t−1) × PD. Calculez `ecl_etape1` (année 1 seulement) et
`ecl_etape2` (5 ans), chaque perte annuelle étant actualisée par (1 + taux)^t.

In [ ]:
# VOTRE CODE ICI
ecl_etape1, ecl_etape2 = ..., ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
_args = (np.array([0.02]), np.array([0.45]), np.array([10_000.0]), np.array([60]))
verifier_proche(ecl_etape1, ifrs9.ecl(*_args, etape=np.array([1]))[0], 1e-3)
verifier_proche(ecl_etape2, ifrs9.ecl(*_args, etape=np.array([2]))[0], 1e-3)

## Bilan

- La fiche de modèle rassemble usage prévu, données, performance, limites et décisions : c'est le dossier de validation.
- L'équité se juge sur les erreurs subies par les légitimes, pas seulement sur les taux de blocage.
- WOE/IV, grille de points et ECL IFRS 9 : le même raisonnement probabiliste, appliqué au crédit (équivalence SAS fournie).

**Pour aller plus loin :**

- `sas/` : la même grille en SAS; `artifacts/credit/equivalence_python_sas.json`.
- BSIF, ligne directrice E-23 (gestion du risque de modèle); IFRS 9, section 5.5.

---
*Fraude-360 — données entièrement synthétiques. Cours conçu par Noé Carème Fouotsa Manfouo · licence MIT (code) et CC BY 4.0 (contenu pédagogique).*